# Topic Modeling: LDA via Gibbs Sampling

Решение задачи выделения тем (Topic Modeling) на датасете 20 Newsgroups с использованием алгоритма Latent Dirichlet Allocation (LDA) и сэмплирования Гиббса.

In [ ]:
import numpy as np
import time
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

### 1. Загрузка и предобработка данных
Загружаем обучающую выборку, удаляя заголовки и цитаты. Используем `CountVectorizer` с фильтрацией редких и слишком частых слов (`min_df`, `max_df`) для улучшения качества тем.

In [ ]:
print("Загрузка данных...")
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
docs = newsgroups.data

print("Векторизация...")
vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    max_df=0.95,  
    min_df=0.01   
)
X = vectorizer.fit_transform(docs)
feature_names = vectorizer.get_feature_names_out()

V = len(feature_names)
M = X.shape[0]

print(f"Размер словаря (V): {V}")
print(f"Количество документов (M): {M}")

### 2. Инициализация параметров LDA
Разворачиваем разреженную матрицу в массивы `doc_ids` и `word_ids` для быстрого доступа. Инициализируем счетчики `n_dk` (документ-тема), `n_kw` (тема-слово) и `n_k` (всего слов в теме).

In [ ]:
doc_ids, word_ids = X.nonzero()
counts = X.data

doc_ids = np.repeat(doc_ids, counts)
word_ids = np.repeat(word_ids, counts)
n_tokens = len(doc_ids)

print(f"Всего токенов: {n_tokens}")


K = 20          
alpha = 1.0
beta = 1.0
n_iters = 50   

z = np.random.randint(0, K, size=n_tokens)

n_dk = np.zeros((M, K), dtype=int)
n_kw = np.zeros((K, V), dtype=int)
n_k = np.zeros(K, dtype=int)

for i in range(n_tokens):
    d = doc_ids[i]
    w = word_ids[i]
    topic = z[i]
    
    n_dk[d, topic] += 1
    n_kw[topic, w] += 1
    n_k[topic] += 1

### 3. Запуск Gibbs Sampling
Основной цикл: для каждого слова убираем его текущую тему из статистики, вычисляем вероятности новых тем и сэмплируем новое значение.

In [ ]:
print(f"Запуск Gibbs Sampling ({n_iters} итераций)...")
start_time = time.time()

for it in range(n_iters):
    for i in range(n_tokens):
        d = doc_ids[i]
        w = word_ids[i]
        old_topic = z[i]
        
        n_dk[d, old_topic] -= 1
        n_kw[old_topic, w] -= 1
        n_k[old_topic] -= 1
        
        
        numer_d = n_dk[d] + alpha
        numer_w = n_kw[:, w] + beta
        denom = n_k + V * beta
        
        p = (numer_d * numer_w) / denom
        p = p / p.sum()  
        
        new_topic = np.random.choice(K, p=p)
        
        z[i] = new_topic
        n_dk[d, new_topic] += 1
        n_kw[new_topic, w] += 1
        n_k[new_topic] += 1
        
    if (it + 1) % 10 == 0:
        print(f"Итерация {it + 1}/{n_iters} завершена. ({time.time() - start_time:.2f} сек)")

### 4. Вывод результатов
Вывод топ-10 слов для каждой из 20 тем.

In [ ]:
print("\n=== Результаты LDA (Топ-10 слов по темам) ===\n")
for k in range(K):
    top_indices = n_kw[k].argsort()[::-1][:10]
    top_words = [feature_names[i] for i in top_indices]
    print(f"Topic {k:02d}: {', '.join(top_words)}")